In [1]:
# %%
# =========================================================
# FEDERATED LEARNING - FedAvg (Flower) untuk Dataset Drebin
# PERBANDINGAN KINERJA: IID vs NON-IID
# =========================================================
# Skrip ini menjalankan simulasi FedAvg DUA KALI dengan konfigurasi
# yang identik (model, jumlah client, jumlah round, dsb.), hanya
# berbeda pada cara data dipartisi ke client:
#   1. IID       -> data dibagi acak merata ke semua client
#   2. Non-IID   -> data dibagi timpang (label-skew) via Dirichlet
#
# Di akhir, kedua hasil (akurasi, precision, recall, f1, loss per round)
# dikumpulkan lalu dibandingkan dalam tabel & grafik.
#
# Dipaksa berjalan di CPU saja (tidak pakai GPU).
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
from typing import Dict, List, Tuple
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
 
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import ndarrays_to_parameters, Context, NDArrays, Scalar
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
# %%
# =========================================================
# Konfigurasi Device - DIPAKSA CPU
# =========================================================
DEVICE = torch.device("cpu")
print(f"[INFO] Device yang digunakan: {DEVICE}")
print("[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).")

2026-09-01 17:43:44,894	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


[INFO] Device yang digunakan: cpu
[INFO] Menjalankan di CPU (dipaksa, GPU tidak digunakan).


In [2]:

# %%
# %% [markdown]
# ## 1. Load & Bersihkan Dataset
 
# %%
DATA_PATH = "../dataset/drebin_dataset.csv"
 
data_raw = pd.read_csv(DATA_PATH)
data = data_raw.copy()
 
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])
 
SPECIAL_COLUMN = "TelephonyManager.getSimCountryIso"
if SPECIAL_COLUMN in data.columns:
    data[SPECIAL_COLUMN] = data[SPECIAL_COLUMN].replace("?", np.nan)
    data[SPECIAL_COLUMN] = pd.to_numeric(data[SPECIAL_COLUMN], errors="coerce")
 
if "class" not in data.columns:
    raise KeyError(f"Kolom 'class' tidak ditemukan. Kolom tersedia: {list(data.columns)}")
 
data["class"] = data["class"].replace({"B": 0, "S": 1})
 
feature_columns = [c for c in data.columns if c != "class"]
data[feature_columns] = data[feature_columns].apply(pd.to_numeric, errors="coerce")
 
data = data.dropna().reset_index(drop=True)
 
X = data.drop(columns=["class"]).values.astype(np.float32)
y = data["class"].astype(int).values
 
print("Ukuran X:", X.shape, "| Ukuran y:", y.shape)
print("Distribusi kelas:", np.bincount(y))
 
# %% [markdown]
# ## 2. Train/Test Split Global + Standardisasi
 
# %%
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
 
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)
 
N_FEATURES = X_train_full.shape[1]
N_CLASSES = 2
 
# %%
# %% [markdown]
# ## 3. Fungsi Partisi Data: IID dan Non-IID
 
# %%
def partition_iid(
    X: np.ndarray, y: np.ndarray, num_clients: int, seed: int = SEED
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Bagi data secara acak merata (IID) ke semua client."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    rng.shuffle(idx)
    splits = np.array_split(idx, num_clients)
    return [(X[s], y[s]) for s in splits]
 
 
def partition_non_iid_dirichlet(
    X: np.ndarray,
    y: np.ndarray,
    num_clients: int,
    alpha: float = 0.3,
    seed: int = SEED,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Bagi data secara non-IID menggunakan distribusi Dirichlet per kelas.
    alpha kecil (misal 0.1-0.5) -> distribusi label makin timpang antar client.
    alpha besar (misal 10+) -> mendekati IID.
    """
    rng = np.random.default_rng(seed)
    n_classes = len(np.unique(y))
    client_indices = [[] for _ in range(num_clients)]
 
    for c in range(n_classes):
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
 
        proportions = rng.dirichlet(alpha=np.repeat(alpha, num_clients))
        cut_points = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, cut_points)
 
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())
 
    partitions = []
    for indices in client_indices:
        indices = np.array(indices, dtype=int)
        rng.shuffle(indices)
        partitions.append((X[indices], y[indices]))
 
    return partitions
 
 
def show_partition_stats(partitions, name="Partition"):
    print("=" * 70)
    print(f" {name}")
    print("=" * 70)
 
    total_data = sum(len(y_client) for _, y_client in partitions)
 
    for client_id, (X_client, y_client) in enumerate(partitions):
        unique, counts = np.unique(y_client, return_counts=True)
        label_counts = dict(zip(unique, counts))
 
        count_0 = label_counts.get(0, 0)
        count_1 = label_counts.get(1, 0)
 
        pct_0 = count_0 / len(y_client) * 100
        pct_1 = count_1 / len(y_client) * 100
 
        print(
            f"Client {client_id + 1}: "
            f"Total={len(y_client):5d} | "
            f"Label 0={count_0:5d} ({pct_0:6.2f}%) | "
            f"Label 1={count_1:5d} ({pct_1:6.2f}%)"
        )
 
    print("-" * 70)
    print(f"Total data : {total_data}")
    print("=" * 70)

Ukuran X: (15031, 215) | Ukuran y: (15031,)
Distribusi kelas: [9476 5555]


C:\Users\s2\AppData\Local\Temp\ipykernel_6868\851256654.py:8: DtypeWarning: Columns (0: TelephonyManager.getSimCountryIso) have mixed types. Specify dtype option on import or set low_memory=False.
  data_raw = pd.read_csv(DATA_PATH)


In [3]:

# %% [markdown]
# ## 4. Konfigurasi Eksperimen
 
# %%
NUM_CLIENTS = 5
NUM_ROUNDS = 10
LOCAL_EPOCHS = 50
BATCH_SIZE = 32
DIRICHLET_ALPHA = 0.3
 
# %% [markdown]
# ## 5. Model PyTorch (MLP sederhana untuk klasifikasi biner)
 
# %%
class DrebinMLP(nn.Module):
    def __init__(self, n_features: int, n_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )
 
    def forward(self, x):
        return self.net(x)
 
 
def get_parameters(model: nn.Module) -> NDArrays:
    return [val.cpu().numpy() for _, val in model.state_dict().items()]
 
 
def set_parameters(model: nn.Module, parameters: NDArrays) -> None:
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    model.load_state_dict(state_dict, strict=True)
 
 
def train(model: nn.Module, X: np.ndarray, y: np.ndarray, epochs: int, batch_size: int, device: torch.device = DEVICE):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
 
 
def test(model: nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 64, device: torch.device = DEVICE) -> Tuple[float, Dict[str, float]]:
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
 
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(X_t, y_t)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
 
    total_loss, correct, total = 0.0, 0, 0
    all_preds: List[int] = []
    all_labels: List[int] = []
 
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * len(yb)
            preds = torch.argmax(out, dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(yb.cpu().numpy().tolist())
 
    avg_loss = total_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
 
    n_unique = len(set(all_labels))
    avg_mode = "binary" if n_unique > 1 else "macro"
 
    precision = precision_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
    recall = recall_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=avg_mode, pos_label=1, zero_division=0)
 
    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
    }
    return avg_loss, metrics
 
 
# %% [markdown]
# ## 6. Flower Client (generik, menerima partisi lewat closure)
 
# %%
class DrebinClient(NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val, device: torch.device = DEVICE):
        self.model = DrebinMLP(N_FEATURES, N_CLASSES)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.device = device
 
    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        return get_parameters(self.model)
 
    def fit(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        train(
            self.model, self.X_train, self.y_train,
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, device=self.device,
        )
        return get_parameters(self.model), len(self.X_train), {}
 
    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):
        set_parameters(self.model, parameters)
        loss, metrics = test(self.model, self.X_val, self.y_val, device=self.device)
        return float(loss), len(self.X_val), metrics
 
 
def make_client_fn(client_partitions):
    """Membuat client_fn yang terikat ke satu set partisi tertentu (IID atau Non-IID)."""
    def client_fn(context: Context) -> Client:
        partition_id = int(context.node_config["partition-id"])
        X_client, y_client = client_partitions[partition_id]
 
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_client, y_client, test_size=0.2, random_state=SEED,
            stratify=y_client if len(np.unique(y_client)) > 1 else None,
        )
        return DrebinClient(X_tr, y_tr, X_val, y_val, device=DEVICE).to_client()
 
    return client_fn
 
 
def weighted_average(metrics: List[Tuple[int, Dict[str, Scalar]]]) -> Dict[str, Scalar]:
    """Agregasi weighted-average untuk accuracy, precision, recall, f1 dari semua client."""
    total_examples = sum(num_examples for num_examples, _ in metrics)
    aggregated: Dict[str, Scalar] = {}
    for key in ["accuracy", "precision", "recall", "f1_score"]:
        weighted_sum = sum(num_examples * m[key] for num_examples, m in metrics if key in m)
        aggregated[key] = weighted_sum / total_examples if total_examples > 0 else 0.0
    return aggregated
 
 
def make_evaluate_fn(X_test: np.ndarray, y_test: np.ndarray, history: List[Dict[str, float]], label: str):
    """
    Membuat evaluate_fn untuk server yang, selain mengevaluasi model global,
    juga MENYIMPAN metrik tiap round ke dalam `history` (dipakai untuk
    perbandingan IID vs Non-IID di akhir).
    """
    def evaluate(server_round: int, parameters: NDArrays, config: Dict[str, Scalar]):
        model = DrebinMLP(N_FEATURES, N_CLASSES)
        set_parameters(model, parameters)
        loss, metrics = test(model, X_test, y_test, device=DEVICE)
        print(
            f"[{label} | Round {server_round}] "
            f"loss={loss:.4f} | acc={metrics['accuracy']:.4f} | "
            f"precision={metrics['precision']:.4f} | recall={metrics['recall']:.4f} | "
            f"f1={metrics['f1_score']:.4f}"
        )
        history.append({
            "mode": label,
            "round": server_round,
            "loss": loss,
            **metrics,
        })
        return loss, metrics
 
    return evaluate

In [4]:

# %% [markdown]
# ## 7. Fungsi untuk Menjalankan Satu Eksperimen (IID atau Non-IID)
 
# %%
def run_experiment(mode: str) -> pd.DataFrame:
    """
    Menjalankan simulasi FedAvg penuh untuk satu mode partisi ("iid" atau "non_iid"),
    lalu mengembalikan DataFrame berisi metrik evaluasi global per round.
    """
    assert mode in ("iid", "non_iid")
    label = "IID" if mode == "iid" else f"Non-IID (alpha={DIRICHLET_ALPHA})"
 
    if mode == "iid":
        partitions = partition_iid(X_train_full, y_train_full, NUM_CLIENTS, seed=SEED)
    else:
        partitions = partition_non_iid_dirichlet(
            X_train_full, y_train_full, NUM_CLIENTS, alpha=DIRICHLET_ALPHA, seed=SEED
        )
 
    show_partition_stats(partitions, name=f"PARTISI - {label}")
 
    history: List[Dict[str, float]] = []
 
    client_app = ClientApp(client_fn=make_client_fn(partitions))
 
    def server_fn(context: Context) -> ServerAppComponents:
        initial_model = DrebinMLP(N_FEATURES, N_CLASSES)
        initial_parameters = ndarrays_to_parameters(get_parameters(initial_model))
 
        strategy = FedAvg(
            fraction_fit=1.0,
            fraction_evaluate=1.0,
            min_fit_clients=NUM_CLIENTS,
            min_evaluate_clients=NUM_CLIENTS,
            min_available_clients=NUM_CLIENTS,
            initial_parameters=initial_parameters,
            evaluate_fn=make_evaluate_fn(X_test, y_test, history, label),
            evaluate_metrics_aggregation_fn=weighted_average,
            fit_metrics_aggregation_fn=weighted_average,
        )
        config = ServerConfig(num_rounds=NUM_ROUNDS)
        return ServerAppComponents(strategy=strategy, config=config)
 
    server_app = ServerApp(server_fn=server_fn)
 
    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": 0.0}}
 
    print(f"\n{'#' * 70}\n# MENJALANKAN SIMULASI: {label}\n{'#' * 70}")
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=NUM_CLIENTS,
        backend_config=backend_config,
    )
 
    return pd.DataFrame(history)
 
 
# %% [markdown]
# ## 8. Jalankan Kedua Eksperimen & Bandingkan
 
# %%
if __name__ == "__main__":
    df_iid = run_experiment("iid")
    df_non_iid = run_experiment("non_iid")
 
    df_all = pd.concat([df_iid, df_non_iid], ignore_index=True)
 
    # =========================================================
    # Tabel hasil per round (semua metrik, kedua mode)
    # =========================================================
    print("\n" + "=" * 90)
    print("HASIL PER ROUND - IID vs NON-IID")
    print("=" * 90)
    print(
        df_all[["mode", "round", "loss", "accuracy", "precision", "recall", "f1_score"]]
        .to_string(index=False)
    )
 
    # =========================================================
    # Ringkasan: metrik pada round terakhir (kondisi akhir training)
    # =========================================================
    final_round = df_all["round"].max()
    df_final = df_all[df_all["round"] == final_round].copy()
 
    print("\n" + "=" * 90)
    print(f"RINGKASAN KINERJA AKHIR (Round {final_round})")
    print("=" * 90)
    print(
        df_final[["mode", "loss", "accuracy", "precision", "recall", "f1_score"]]
        .to_string(index=False)
    )
 
    # =========================================================
    # Ringkasan: rata-rata metrik selama seluruh training (semua round)
    # =========================================================
    df_avg = df_all.groupby("mode")[["loss", "accuracy", "precision", "recall", "f1_score"]].mean()
 
    print("\n" + "=" * 90)
    print(f"RATA-RATA KINERJA SELAMA {NUM_ROUNDS} ROUND")
    print("=" * 90)
    print(df_avg.to_string())
 
    # =========================================================
    # Simpan hasil ke CSV untuk dianalisis / diplot lebih lanjut
    # =========================================================
    output_path = "hasil_perbandingan_iid_vs_noniid.csv"
    df_all.to_csv(output_path, index=False)
    print(f"\n[INFO] Hasil lengkap per round disimpan ke: {output_path}")
 
    # =========================================================
    # (Opsional) Grafik perbandingan akurasi & f1-score per round
    # =========================================================
    try:
        import matplotlib.pyplot as plt
 
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
        for label, df_mode in df_all.groupby("mode"):
            axes[0].plot(df_mode["round"], df_mode["accuracy"], marker="o", label=label)
            axes[1].plot(df_mode["round"], df_mode["f1_score"], marker="o", label=label)
 
        axes[0].set_title("Akurasi Global per Round")
        axes[0].set_xlabel("Round")
        axes[0].set_ylabel("Accuracy")
        axes[0].legend()
        axes[0].grid(alpha=0.3)
 
        axes[1].set_title("F1-Score Global per Round")
        axes[1].set_xlabel("Round")
        axes[1].set_ylabel("F1-Score")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
 
        plt.tight_layout()
        fig_path = "perbandingan_iid_vs_noniid.png"
        plt.savefig(fig_path, dpi=150)
        print(f"[INFO] Grafik perbandingan disimpan ke: {fig_path}")
    except ImportError:
        print("[INFO] matplotlib tidak tersedia, grafik dilewati (tabel/CSV tetap tersimpan).")
 

 PARTISI - IID
Client 1: Total= 2405 | Label 0= 1525 ( 63.41%) | Label 1=  880 ( 36.59%)
Client 2: Total= 2405 | Label 0= 1501 ( 62.41%) | Label 1=  904 ( 37.59%)
Client 3: Total= 2405 | Label 0= 1523 ( 63.33%) | Label 1=  882 ( 36.67%)
Client 4: Total= 2405 | Label 0= 1477 ( 61.41%) | Label 1=  928 ( 38.59%)
Client 5: Total= 2404 | Label 0= 1554 ( 64.64%) | Label 1=  850 ( 35.36%)
----------------------------------------------------------------------
Total data : 12024

######################################################################
# MENJALANKAN SIMULASI: IID
######################################################################


INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Evaluating initial global parameters
INFO :      initial parameters (loss, other metrics): 0.7000424462297171, {'accuracy': 0.504822081809112, 'precision': 0.426056338028169, 'recall': 0.9801980198019802, 'f1_score': 0.5939460049086447}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)


[IID | Round 0] loss=0.7000 | acc=0.5048 | precision=0.4261 | recall=0.9802 | f1=0.5939


(raylet) Stack (most recent call first):
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 1170 in wrapper
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py", line 627Windows fatal exception:  in job_logging_config
(raylet)   File "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py"2807 in disconnect
(raylet)   File ", line 134 in wrapper
(raylet) 
(raylet) 
(raylet)   File Windows fatal exception: access violation
(raylet) 
(raylet) "c:\Users\s2\Documents\FedMal\.fed\Lib\site-packages\ray\_private\worker.py"Stack (most 

[IID | Round 1] loss=0.0891 | acc=0.9814 | precision=0.9656 | recall=0.9847 | f1=0.9750


...

[Non-IID (alpha=0.3) | Round 9] loss=0.1441 | acc=0.9880 | precision=0.9812 | recall=0.9865 | f1=0.9838


INFO :      aggregate_evaluate: received 5 results and 0 failures
INFO :      
INFO :      [ROUND 10]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      fit progress: (10, 0.16306903608333265, {'accuracy': 0.9886930495510475, 'precision': 0.9846984698469847, 'recall': 0.9846984698469847, 'f1_score': 0.9846984698469847}, 116.33844409999438)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)


[Non-IID (alpha=0.3) | Round 10] loss=0.1631 | acc=0.9887 | precision=0.9847 | recall=0.9847 | f1=0.9847


INFO :      aggregate_evaluate: received 5 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 10 round(s) in 117.34s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.10830286703229644
INFO :      		round 2: 0.16170945628670044
INFO :      		round 3: 0.14418413532815189
INFO :      		round 4: 0.16660466582190356
INFO :      		round 5: 0.17677815999491805
INFO :      		round 6: 0.1915748467669278
INFO :      		round 7: 0.19327176280214403
INFO :      		round 8: 0.18100816952060764
INFO :      		round 9: 0.20549672790485607
INFO :      		round 10: 0.22911646966508423
INFO :      	History (loss, centralized):
INFO :      		round 0: 0.686082620734902
INFO :      		round 1: 0.07880086865800141
INFO :      		round 2: 0.09202038994264
INFO :      		round 3: 0.09674841096227674
INFO :      		round 4: 0.11064002201262159
INFO :      		round 5: 0.1320065659575231
INFO :      		round 6: 0.13809361366606507
INFO :      		round 7: 0.14237845310429761


HASIL PER ROUND - IID vs NON-IID
               mode  round     loss  accuracy  precision   recall  f1_score
                IID      0 0.700042  0.504822   0.426056 0.980198  0.593946
                IID      1 0.089074  0.981377   0.965578 0.984698  0.975045
                IID      2 0.085053  0.984037   0.973286 0.983798  0.978514
                IID      3 0.078529  0.987363   0.982031 0.983798  0.982914
                IID      4 0.078253  0.987695   0.982047 0.984698  0.983371
                IID      5 0.099227  0.985367   0.977619 0.982898  0.980251
                IID      6 0.092323  0.986698   0.979409 0.984698  0.982047
                IID      7 0.110476  0.986365   0.981982 0.981098  0.981540
                IID      8 0.091189  0.987695   0.980322 0.986499  0.983401
                IID      9 0.090517  0.985700   0.977639 0.983798  0.980709
                IID     10 0.090008  0.986698   0.977698 0.986499  0.982079
Non-IID (alpha=0.3)      0 0.686083  0.593615   0.1275